# **Instalar dependencias**

In [ ]:
# Verificar GPU
import torch
import os
from ultralytics import YOLO
import cv2
import numpy
import natsort


print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

# **Hacer la prediccion de una y varias imagenes**

In [ ]:
# Load a model
model = YOLO("yolo12n.pt")

In [ ]:
img_path = 'examples'

path_img1 = os.path.join(img_path, 'pedestrian.jpg')
path_img2 = os.path.join(img_path, 'colibri0018.jpg')
path_img3 = os.path.join(img_path, 'mora0006.jpg')
# Run batched inference on a list of images
# Pueden colocar una lista de imagenes (paths de preferencia para no saturar la RAM)
# La clases 0 en coco pertenece a la clase persona.

results = model([path_img1, path_img3], classes = [0, 14, 32])  # return a list of Results objects.
# Para el caso de BIRD, dado que la imagen es de baja calidad y relación de aspecto desigual, es preferible bajar la confidencia
#para encontrar la detección, pero solo para este caso

#______________________________________________________
results_sephanoides = model([path_img2], conf = 0.1)
#______________________________________________________

# Process results list
# for result in results:
#     boxes = result.boxes  # Boxes object for bounding box outputs
#     result.show()  # display to screen

# for result in results_sephanoides:
#     boxes = result.boxes  # Boxes object for bounding box outputs
#     result.show()  # display to screen


## Detector + Tracking
Aplique el mismo método anterior sobre los frames de las carpetas “estacionamiento”, “mora_desestabilizado” y “mora_mov”. Haga una comparación con las obtenidas en el punto 5. Finalmente genere videos en formato MP4 de con una tasa de 25 fps y concluya

In [ ]:
img_path = 'Lab-Mov'
estacionamiento = os.path.join(img_path, 'estacionamiento')
mora_desestabilizado = os.path.join(img_path, 'mora_desestabilizado')
mora_mov = os.path.join(img_path, 'mora_mov')
results_path = 'Results_YOLO'

def img_lists(img_path):
    """ Entrega una lista de los path de unas imagenes
    """
    img_list = []
    for img in os.listdir(img_path):
        if img.endswith(('.bmp', '.jpg')):
            img_list.append(os.path.join(img_path, img)) 

    return img_list

estacionamiento_imgs = img_lists(estacionamiento)
mora_mov_imgs = img_lists(mora_mov)
mora_desestabilizado_imgs = img_lists(mora_desestabilizado)


def save_yolo_results_as_images(model, img_list, output_folder, use_gpu=True):
    """
    Procesa imágenes una por una con YOLO y guarda los resultados (con bounding boxes).
    """
    device = 0 if (torch.cuda.is_available() and use_gpu) else 'cpu'
    os.makedirs(output_folder, exist_ok=True)
    print(f"Guardando resultados en: {output_folder}")
    print(f"Usando dispositivo: {device}")

    for i, img_path in enumerate(img_list):
        results = model(img_path, classes=[0, 14, 32], device=device, verbose=False)
        frame = results[0].plot()  # imagen con las detecciones
        frame_bgr = frame

        # nombre de salida igual al original pero en jpg
        name = os.path.splitext(os.path.basename(img_path))[0] + '.jpg'
        save_path = os.path.join(output_folder, name)
        cv2.imwrite(save_path, frame_bgr)

        # liberar memoria GPU
        del results
        torch.cuda.empty_cache()

        if i % 20 == 0:
            print(f"Procesadas {i+1}/{len(img_list)} imágenes...")

    print("Todas las imágenes con detecciones fueron guardadas.")

mora_mov_results_path = os.path.join(results_path, 'mora_mov_frames')
mora_desestabilizado_results_path = os.path.join(results_path, 'mora_desestabilizado_frames')
estacionamiento_results_path = os.path.join(results_path, 'estacionamiento_frames')


# Paso 1: guardar imágenes con detecciones
save_yolo_results_as_images(model, mora_mov_imgs, mora_mov_results_path, use_gpu=True)
save_yolo_results_as_images(model, mora_desestabilizado_imgs, mora_desestabilizado_results_path, use_gpu=True)
save_yolo_results_as_images(model, estacionamiento_imgs, estacionamiento_results_path, use_gpu=True)



In [ ]:

def create_video_from_images(image_folder, output_path, fps=25):
    """
    Crea un video a partir de imágenes ordenadas naturalmente.
    """
    images = [img for img in os.listdir(image_folder) if img.endswith(('.jpg', '.png', '.bmp'))]
    images = natsort.natsorted(images)  # orden natural tipo frame_1, frame_2...

    if not images:
        print(f"No hay imágenes en {image_folder}")
        return

    first_frame = cv2.imread(os.path.join(image_folder, images[0]))
    h, w, _ = first_frame.shape
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    video = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

    for img_name in images:
        img_path = os.path.join(image_folder, img_name)
        frame = cv2.imread(img_path)
        video.write(frame)

    video.release()
    print(f"Video guardado en: {output_path}")

# Paso 2: generar video desde esas imágenes
mora_mov_video_path = os.path.join(results_path, 'mora_mov.mp4')
mora_desestabilizado_video_path = os.path.join(results_path, 'mora_desestabilizado.mp4')
estacionamiento_video_path = os.path.join(results_path, 'estacionamiento.mp4')

create_video_from_images(mora_mov_results_path, mora_mov_video_path, fps=25)
create_video_from_images(estacionamiento_results_path, estacionamiento_video_path, fps=25)
create_video_from_images(mora_desestabilizado_results_path, mora_desestabilizado_video_path, fps=25)
